In [1]:
using YAML
using PrettyPrint
using OffsetArrays
using Revise
Revise.revise()
using BenchmarkTools

In [2]:
using CovidSim_ilm

In [3]:
cs = CovidSim_ilm

CovidSim_ilm

In [4]:
cd(joinpath(homedir(),"Dropbox/Covid Modeling/Covid-ILM/source"))

In [5]:
paramdir = "../parameters"
variantfilename = "variants.yml"
infectdict = YAML.load_file(joinpath(paramdir, variantfilename))

Dict{Any, Any} with 6 entries:
  "usevariants"  => ["base"]
  "base"         => Dict{Any, Any}("spread"=>Dict{Any, Any}("sendrisk"=>[0.0, 0…
  "omicron"      => Dict{Any, Any}("spread"=>Dict{Any, Any}("sendrisk"=>Any[], …
  "loadvariants" => ["base", "alpha", "delta", "omicron"]
  "alpha"        => Dict{Any, Any}("spread"=>Dict{Any, Any}("sendrisk"=>Any[], …
  "delta"        => Dict{Any, Any}("spread"=>Dict{Any, Any}("sendrisk"=>Any[], …

In [6]:
ra = infectdict["omicron"]["transition"]["factors"]["riskadjust"]

6-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.1
 1.1
 1.1

In [7]:
tr = infectdict["base"]["transition"]["tree"]

Dict{Any, Any} with 5 entries:
  "age0_19"  => Dict{Any, Any}(5=>Dict{Any, Any}("duration"=>25, "transition"=>D…
  "age40_59" => Dict{Any, Any}(5=>Dict{Any, Any}("duration"=>25, "transition"=>D…
  "age20_39" => Dict{Any, Any}(5=>Dict{Any, Any}("duration"=>25, "transition"=>D…
  "age60_79" => Dict{Any, Any}(5=>Dict{Any, Any}("duration"=>25, "transition"=>D…
  "age80_up" => Dict{Any, Any}(5=>Dict{Any, Any}("duration"=>25, "transition"=>D…

In [8]:
keys(sort(tr))

KeySet for a OrderedCollections.OrderedDict{Any, Any} with 5 entries. Keys:
  "age0_19"
  "age20_39"
  "age40_59"
  "age60_79"
  "age80_up"

In [9]:
sort(tr["age0_19"])

OrderedCollections.OrderedDict{Any, Any} with 5 entries:
  1 => Dict{Any, Any}("duration"=>5, "transition"=>Dict{Any, Any}("sick"=>[0.0, …
  2 => Dict{Any, Any}("duration"=>9, "transition"=>Dict{Any, Any}("sick"=>[0.0, …
  3 => Dict{Any, Any}("duration"=>14, "transition"=>Dict{Any, Any}("sick"=>[0.85…
  4 => Dict{Any, Any}("duration"=>19, "transition"=>Dict{Any, Any}("sick"=>[0.0,…
  5 => Dict{Any, Any}("duration"=>25, "transition"=>Dict{Any, Any}("sick"=>[0.97…

In [10]:
tr["age0_19"][1]["transition"]

Dict{Any, Any} with 4 entries:
  "sick"   => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  "nil"    => [0.0, 0.4, 0.5, 0.1, 0.0, 0.0]
  "severe" => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  "mild"   => [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

In [11]:

age = first(sort(tr))[2]


Dict{Any, Any} with 5 entries:
  5 => Dict{Any, Any}("duration"=>25, "transition"=>Dict{Any, Any}("sick"=>[0.97…
  4 => Dict{Any, Any}("duration"=>19, "transition"=>Dict{Any, Any}("sick"=>[0.0,…
  2 => Dict{Any, Any}("duration"=>9, "transition"=>Dict{Any, Any}("sick"=>[0.0, …
  3 => Dict{Any, Any}("duration"=>14, "transition"=>Dict{Any, Any}("sick"=>[0.85…
  1 => Dict{Any, Any}("duration"=>5, "transition"=>Dict{Any, Any}("sick"=>[0.0, …

In [12]:

# age = first(sort(tr))[2]


prepdict = Dict(age_key => [cs.Transitiondef(brk["duration"], 
                                             vcat(brk["transition"]["nil"]',brk["transition"]["mild"]',
                                                brk["transition"]["sick"]',brk["transition"]["severe"]')
                                            ) 
                            for (_, brk) in sort(age)] 
                    for (age_key, age) in sort(tr))


Dict{String, Vector{CovidSim_ilm.Transitiondef}} with 5 entries:
  "age0_19"  => [Transitiondef(5, [0.0 0.4 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.…
  "age40_59" => [Transitiondef(5, [0.0 0.2 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.…
  "age20_39" => [Transitiondef(5, [0.0 0.2 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.…
  "age60_79" => [Transitiondef(5, [0.0 0.15 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0…
  "age80_up" => [Transitiondef(5, [0.0 0.1 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.…

In [13]:
at = cs.Agetree(prepdict["age0_19"], prepdict["age20_39"], prepdict["age40_59"], prepdict["age60_79"], prepdict["age80_up"])

CovidSim_ilm.Agetree(CovidSim_ilm.Transitiondef[CovidSim_ilm.Transitiondef(5, [0.0 0.4 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]), CovidSim_ilm.Transitiondef(9, [0.9 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.05 0.0; 0.0 0.0 … 0.0 0.0]), CovidSim_ilm.Transitiondef(14, [1.0 0.0 … 0.0 0.0; 1.0 0.0 … 0.0 0.0; 0.85 0.0 … 0.03 0.0; 0.692 0.0 … 0.302 0.006]), CovidSim_ilm.Transitiondef(19, [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.891 0.0 … 0.106 0.003]), CovidSim_ilm.Transitiondef(25, [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.976 0.0 … 0.0 0.024; 0.91 0.0 … 0.0 0.09])], CovidSim_ilm.Transitiondef[CovidSim_ilm.Transitiondef(5, [0.0 0.2 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0]), CovidSim_ilm.Transitiondef(9, [0.9 0.0 … 0.0 0.0; 0.85 0.0 … 0.0 0.0; 0.0 0.0 … 0.1 0.0; 0.0 0.0 … 0.0 0.0]), CovidSim_ilm.Transitiondef(14, [1.0 0.0 … 0.0 0.0; 1.0 0.0 … 0.0 0.0; 0.83 0.0 … 0.07 0.0; 0.474 0.0 … 0.514 0.012]), CovidSim_ilm.Trans

In [14]:
fieldnames(typeof(at))

(:age0_19, :age20_39, :age40_59, :age60_79, :age80_up)

In [15]:
at.age0_19

5-element Vector{CovidSim_ilm.Transitiondef}:
 CovidSim_ilm.Transitiondef(5, [0.0 0.4 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0])
 CovidSim_ilm.Transitiondef(9, [0.9 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.05 0.0; 0.0 0.0 … 0.0 0.0])
 CovidSim_ilm.Transitiondef(14, [1.0 0.0 … 0.0 0.0; 1.0 0.0 … 0.0 0.0; 0.85 0.0 … 0.03 0.0; 0.692 0.0 … 0.302 0.006])
 CovidSim_ilm.Transitiondef(19, [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.891 0.0 … 0.106 0.003])
 CovidSim_ilm.Transitiondef(25, [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.976 0.0 … 0.0 0.024; 0.91 0.0 … 0.0 0.09])

In [52]:
cs.display_tree_struct(at)

age0_19 # field of struct Agetree, values are vectors
    brk: 1 # element of Vector{Transitiondef}
    duration: 5
    transitions: 
        [0.0, 0.391304347826087, 0.4891304347826087, 0.11956521739130435, 0.0, 0.0]
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    brk: 2 # element of Vector{Transitiondef}
    duration: 9
    transitions: 
        [0.9, 0.0, 0.0, 0.1, 0.0, 0.0]
        [0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
        [0.0, 0.0, 0.0, 0.95, 0.05, 0.0]
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
    brk: 3 # element of Vector{Transitiondef}
    duration: 14
    transitions: 
        [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        [1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        [0.85, 0.0, 0.0, 0.12, 0.03, 0.0]
        [0.692, 0.0, 0.0, 0.0, 0.302, 0.006]
    brk: 4 # element of Vector{Transitiondef}
    duration: 19
    transitions: 
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        [0.0, 0.0, 0.0

In [17]:
sum(at.age0_19[1].trarr[1,:] .*= [0.9, 0.9, 0.9, 1.1, 1.1, 1.1])

0.92

In [18]:
k = 1 / sum(at.age0_19[1].trarr[1,:])

1.0869565217391304

In [19]:
at.age0_19[1].trarr[1, :] .*= k

6-element view(::Matrix{Float64}, 1, :) with eltype Float64:
 0.0
 0.391304347826087
 0.4891304347826087
 0.11956521739130435
 0.0
 0.0

In [20]:
sum(at.age0_19[1].trarr[1,:])

1.0

In [21]:
at2 = cs.setup_dt(tr);

In [22]:
at2.age0_19

5-element Vector{CovidSim_ilm.Transitiondef}:
 CovidSim_ilm.Transitiondef(5, [0.0 0.4 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0])
 CovidSim_ilm.Transitiondef(9, [0.9 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.05 0.0; 0.0 0.0 … 0.0 0.0])
 CovidSim_ilm.Transitiondef(14, [1.0 0.0 … 0.0 0.0; 1.0 0.0 … 0.0 0.0; 0.85 0.0 … 0.03 0.0; 0.692 0.0 … 0.302 0.006])
 CovidSim_ilm.Transitiondef(19, [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.891 0.0 … 0.106 0.003])
 CovidSim_ilm.Transitiondef(25, [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; 0.976 0.0 … 0.0 0.024; 0.91 0.0 … 0.0 0.09])

In [23]:
@btime cs.setup_dt($tr);

  37.375 μs (519 allocations: 24.88 KiB)


In [24]:
keys(infectdict["base"])

KeySet for a Dict{Any, Any} with 3 entries. Keys:
  "spread"
  "immunity"
  "transition"

In [25]:
keys(infectdict["base"]["transition"])

KeySet for a Dict{Any, Any} with 2 entries. Keys:
  "factors"
  "tree"

In [26]:
keys(infectdict["base"]["transition"]["tree"])

KeySet for a Dict{Any, Any} with 5 entries. Keys:
  "age0_19"
  "age40_59"
  "age20_39"
  "age60_79"
  "age80_up"

In [27]:
keys(infectdict["base"]["transition"]["tree"]["age0_19"][1])

KeySet for a Dict{Any, Any} with 2 entries. Keys:
  "duration"
  "transition"

In [28]:
keys(infectdict["base"]["transition"]["tree"]["age0_19"][1]["transition"])

KeySet for a Dict{Any, Any} with 4 entries. Keys:
  "sick"
  "nil"
  "severe"
  "mild"

In [29]:
Symbol("age0_19")

:age0_19

In [30]:
pprintln(infectdict["delta"])

{
  "spread" : {
               "sendrisk" : [],
               "recvrisk" : [],
               "infectmultiplier" : 1.2,
             },
  "immunity" : {"immunestrength" : 0.6, 
                "immunehalflife" : 240},
  "transition" : {
                   "factors" : {
                                 "riskadjust" : [1.0, 
                                                 1.0, 
                                                 1.0, 
                                                 1.1, 
                                                 1.1, 
                                                 1.1],
                               },
                   "tree" : nothing,
                 },
}


In [31]:
isnothing(infectdict["delta"]["transition"]["tree"])

true

In [32]:
transactionset, trvec = cs.build_transition_params(infectdict);

In [33]:
trvec

6-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [34]:
keys(transactionset)

KeySet for a Dict{Any, Any} with 4 entries. Keys:
  :alpha
  :base
  :omicron
  :delta

In [35]:
transactionset[:base].riskadjust

Float64[]

In [36]:
transactionset[:omicron].riskadjust

6-element Vector{Float64}:
 1.0
 1.0
 1.0
 1.1
 1.1
 1.1

In [37]:
println(transactionset[:base].tree.age0_19[5].duration)
pprintln(transactionset[:base].tree.age0_19[1].trarr)

25
Matrix{Float64}()


In [38]:
transactionset[:base].tree.age0_19[5].trarr

4×6 Matrix{Float64}:
 0.0    0.0  0.0  0.0  0.0  0.0
 0.0    0.0  0.0  0.0  0.0  0.0
 0.976  0.0  0.0  0.0  0.0  0.024
 0.91   0.0  0.0  0.0  0.0  0.09

In [39]:
infectdict["base"]["spread"]

Dict{Any, Any} with 3 entries:
  "sendrisk"         => [0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.65, 0.6…
  "recvrisk"         => [0.1, 0.39, 0.44, 0.54, 0.56]
  "infectmultiplier" => 1.0

In [40]:
infectdict["base"]["immunity"]

Dict{Any, Any} with 2 entries:
  "immunestrength" => 0.6
  "immunehalflife" => 180

In [41]:
variant = "base"
pass1 = merge(infectdict[variant]["spread"], infectdict[variant]["immunity"])
pass2 = Dict(Symbol(k)=> v for (k,v) in pass1)

Dict{Symbol, Any} with 5 entries:
  :infectmultiplier => 1.0
  :recvrisk         => [0.1, 0.39, 0.44, 0.54, 0.56]
  :immunehalflife   => 180
  :sendrisk         => [0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.65, 0.6 …
  :immunestrength   => 0.6

In [42]:
Base.@kwdef struct Foo
    age0_19::Dict{Any, Any}
end

Foo

In [43]:
f1 = Foo(age0_19=Dict("one"=>1))

Foo(Dict{Any, Any}("one" => 1))

In [44]:
f1.age0_19

Dict{Any, Any} with 1 entry:
  "one" => 1

In [45]:
Base.@kwdef struct Transitionparams
    tree::Union{Dict{agegrp, Dict{Int64, Dict{Symbol, Matrix{Float64}}}}, Nothing}
    riskadjust::Vector{Float64}   # use [] for nothing
end

Transitionparams

In [46]:
tp = Transitionparams(tree=nothing, riskadjust=[])

Transitionparams(nothing, Float64[])

In [53]:
CovidSim_ilm.Infectparams(pass2)

Infectparams([0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.65, 0.6  …  0.05, 0.05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.1, 0.39, 0.44, 0.54, 0.56], 0.6, 180, 1.0)

In [49]:
infectset = CovidSim_ilm.build_spread_params(infectdict)

Dict{Symbol, Infectparams} with 4 entries:
  :alpha   => Infectparams(Float64[], Float64[], 0.6, 240, 1.2)
  :base    => Infectparams([0.0, 0.3, 0.65, 0.75, 0.85, 0.85, 0.75, 0.7, 0.65, …
  :omicron => Infectparams(Float64[], Float64[], 0.6, 240, 1.2)
  :delta   => Infectparams(Float64[], Float64[], 0.6, 240, 1.2)

In [50]:
infectset[:omicron].infectmultiplier

1.2